In [1]:
# import 

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt
from dateutil import tz
from suntime import Sun, SunTimeException

from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/
from utils import process_data
from utils.utilities import to_datetime

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

#save figures in...
dir_save = './output/check_data/'

In [ ]:
## Read all data
%autoreload 2
from input.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


In [ ]:
### Plot all species
# species to plot (all available, or make a selection)
sel_species = ds_all.species
unique_species = np.unique(sel_species)

# moving window
mw = 24 * 10  # hours
# select time period
t1 = "2020-01-01"
t2 = "2023-12-31"


# function to plot each subplot
def plot_data(ds_temp, i_s, mw_temp, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()
    ds_temp.plot(ax=ax, ls="", marker=".", alpha=0.7, label=str(i_s))
    # moving mean
    if ("flask" in i_s) == False:  # no moving mean for flask
        ds_temp.rolling(time=mw_temp, center=True, min_periods=mw_temp / 2).mean().plot(
            ax=ax, ls="-", c="k", label=f"Moving Mean ({int(mw_temp/24)}days)"
        )


## Start figure
fig, axs = plt.subplots(
    len(unique_species),
    1,
    sharex=True,
    figsize=(8, len(unique_species) * 2),
    layout="constrained",
)

for s, ax in zip(unique_species, axs):
    ds_sel = ds_all.where(ds_all.species == s, drop=True).sel(time=slice(t1, t2))
    if len(ds_sel.dataset) > 1:  # several datasets with same species
        for ii in ds_sel.dataset:
            # plot data (adapt moving window for flask)
            plot_data(
                ds_sel.sel(dataset=ii).value,
                ii.item(),
                mw * 3 if ii.astype(str).str.contains("flask") else mw,
                ax,
            )
    else:
        plot_data(ds_sel.sel(dataset=s).value, s, mw, ax)

    ax.set_ylabel(f"{ds_sel.species.values[0]} ({ds_sel.unit.values[0]})")
    ax.set_xlabel("")
    ax.set_title("")
# ax.set_xlim(np.array([t1, t2], dtype="datetime64"))

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper left")
fig.align_ylabels()

plt.suptitle("Mt. Kenya GAW station")

plt.savefig(
    f"{dir_save}timeseries_{t1[0:4]}_{t2[0:4]}.png", dpi=300
)

In [ ]:
# moving window
mw = 24 * 3  # hours

plt.figure()
ds = ds_all.sel(dataset="CO2")
ds.value.plot(ls="", marker=".")

ds.value.dropna("time").rolling(time=mw, center=True, min_periods=mw / 2).mean().plot(
    ls="-"
)

# check QCflag = 3
ds.where(ds.QCflag == 3).value.plot(ls="", marker=".", c="r")


plt.show()

In [ ]:
# remove outliers or not
remove_outliers = False #remove outliers based ont uncertainty and statistics
remove_fire_events = True # remove spcific fire events (defined below)

#-----
fire_dates_Mar2022 = slice('2022-03-16','2022-03-29')
fire_times_Mar2022 = ds_all.sel(time=fire_dates_Mar2022).time.values


fire_dates_Mar2021 = slice('2021-03-20','2021-03-28')
fire_times_Mar2021 = ds_all.sel(time=fire_dates_Mar2021).time.values

if remove_fire_events:
    # remove fire events from the dataset. Use a workaround, because xarray adds time dimension to all variables
    ds_withtime = ds_all.drop([ var for var in ds_all.variables if not 'time' in ds_all[var].dims ])
    ds_timeless = ds_all.drop([ var for var in ds_all.variables if     'time' in ds_all[var].dims ])
    ds_withtime = ds_withtime.where(ds_withtime.time.isin(fire_times_Mar2022)==False) #don't use the fire event times 2022
    ds_withtime = ds_withtime.where(ds_withtime.time.isin(fire_times_Mar2021)==False) #don't use the fire event times 2021
    ds_all_rem_out = xr.merge([ds_timeless, ds_withtime])

if remove_outliers:
    # Remove outliers that exceed 10*stdedeviation_mean and 4* the zscore 
    # remove outliers for each species sperarately
    ds_all_rem_out = ds_all.copy(deep=True) #use deepcopy, otherwise it replaces values in ds_all!
    outlier_masks = {}
    for ds in np.unique(ds_all.species):
        print(f"remove outliers for {ds}:")
        ds_all_rem_out.loc[dict(dataset=ds)], mask_removed_outliers= process_data.rem_out(ds_all.sel(dataset=ds), std_fac=10, z_threshold=4)
        outlier_masks[ds] = mask_removed_outliers

### Baseline with moving percentiles

In [ ]:
# Use CO to determine background-conditions
# moving window for running percentile
mw = 24 * 45  # 24 * days (data given in hours)
quantile_value = 10
show_CO_background = True # show the background conditions as they were defined by CO

t1 = "2020-01-01"
t2 = "2023-12-31"

species = ['CO', 'CO2', 'CH4', 'O3']

fig, axs = plt.subplots(
    int(len(species)/2),
    2,
    sharex=True,
    figsize=(15,8),
    layout="constrained",
)

for s, ax in zip(species, axs.flatten()):
    ds_sel = ds_all_rem_out.sel(dataset=s, time=slice(t1, t2))

    ds_sel.value.plot(marker=".", color="dimgrey", ls="", ax=ax,zorder=0)
    percentiles = (
        ds_sel.value.rolling(time=mw, center=True).construct("tmp").quantile(quantile_value/100, dim="tmp")
    )
    percentiles.plot(
        marker=".", ls="-", color="k", lw=1,ax=ax, label=f'moving {quantile_value}th percentile ({int(mw/24)}days)',zorder=2
    )  # tak3 3days windows and for each window the 10%-quantile

    # plot the background and the difference to this "background":
    mask_is_background = ds_sel.value < percentiles # define all data below the percentile as background
    if s == 'CO':
        mask_is_background_co = mask_is_background

    # plot the "background":
    ds_sel.value[mask_is_background].plot(color='lightgrey',ls='',marker='.',ax=ax,label='background',zorder=1)
    # plot the times where CO detected background conditions: 
    if s != 'CO' and show_CO_background:
        ds_sel.value[mask_is_background_co].plot(color='blue',ls='',marker='.', markersize=3,ax=ax,label='background conditions from CO',zorder=3)

    # plot the remaining data in second axes ("local contamination")
    # delta = ds_sel.value - percentile
    # delta.where(delta > 0).plot(
    #     ax=ax1, color='dimgrey'
    # )  # plot all data that are larger than the "background" in new subplot
    # ax0.set_xlabel('')
    # ax1.set_title("")
    # ax1.set_title("Delta  (data-background)", loc="left")
    
    ax.set_ylabel(f"{ds_sel.species.values} ({ds_sel.unit.values})")
    ax.set_xlabel("")
    ax.set_title("")
    ax.legend()
fig_name = 'timeseries_and_background_percentiles'
if show_CO_background:
    fig_name += '_withCO_background'
#axs.flatten()[0].set_title('',loc='left')
plt.suptitle(f'Background conditions at MKN (selected with {quantile_value}th percentiles)')
plt.show()
plt.savefig(
   f"{dir_save}background/{fig_name}.png", dpi=300
)

In [ ]:
# plot the percentile and the delta for one species

#ds_sel = ds_all.sel(dataset="CO", time=slice(t1, t2))
ds_sel = ds_all_rem_out.sel(dataset="CO", time=slice(t1, t2))
fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax0 = axs[0]
ax1 = axs[1]
ds_sel.value.plot(marker=".", color="dimgrey", ls="", ax=ax0,zorder=0)
percentile_10th = (
    ds_sel.value.rolling(time=mw, center=True).construct("tmp").quantile(quantile_value/100, dim="tmp")
)
percentile_10th.plot(
    marker=".", ls="-", color="k", lw=1,ax=ax0, label=f'moving {quantile_value}th percentile ({int(mw/24)}days)',zorder=2
)  # take 3days windows and for each window the 10%-quantile

# plot the background and the difference to this "background":
mask_is_background_co = ds_sel.value < percentile_10th
# plot the "background":
ds_sel.value[mask_is_background_co].plot(color='lightgrey',ls='',marker='.',ax=ax0,label='background',zorder=1)

#plot the remaining data in second axes ("local contamination")
delta_co = ds_sel.value - percentile_10th
delta_co.where(delta_co > 0).plot(
    ax=ax1, color='dimgrey'
)  # plot all data that are larger than the "background" in new subplot
ax0.set_xlabel('')
ax1.set_title("")
ax1.set_title("Delta CO (data-background)", loc="left")
ax0.legend()
plt.show()

In [8]:
# Define fit paramaters for the curve fitting (NOAA approach)
from utils.ccg_filter import ccg_filter as ccgfilt
from utils.ccg_filter import ccg_dates
from utils import run_curve_fit

# Default values
fit_params_defaults = {'shortterm': 80, #Short term cutoff value in days for smoothing of data
                'longterm': 667, # smoothing in days. Default: 667
                'numpolyterms': 3, # use only 2 for less than 3 years of data, otherwise use 3 (=quadratic fit)
                'sampleinterval': 1 / 24,  # 1h
                'numharmonics': 4}

fit_properties = {}
for dataset in ds_all_rem_out.dataset:
    dataset_name = dataset.values.item()  # Convert numpy array to a hashable type
    fit_properties[dataset_name] = fit_params_defaults.copy()

#### compare different background methods

In [9]:
### Load marine boundary layer data from NOAA for equator
def get_noaa_file(file_path):
    # get the header line: 
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                header = line
            else:
                break #stop when there are no more #

    header = header[1:].strip().split()
    df = pd.read_csv(file_path, 
                            #sep=" ",
                            names= header, 
                            comment='#',
                            delim_whitespace=True, # or use sep='\s+'
                            parse_dates={'time':[0,1,2]},
                            index_col='time',
                            )
    return df


mbl_co2 = get_noaa_file("../data/noaa/co2_mblr_equ.csv")

mbl_ch4 = get_noaa_file("../data/noaa/ch4_mblr_equ.csv")


In [ ]:
from processing import wdc

co_sey = wdc.compile_wdcgg_into_dataframe(data_path = "../data/wdc/wdcgg/SEY/CO",sampling="event")

In [ ]:
plt.figure()
co_sey['value'].plot(marker='.',ls='')
plt.show()

In [38]:
## plot background for a single species, including the NOAA fit and NOAA MBL (marine boundary layer) data

def plot_background(    
        dataset_str: str, 
        t1: str,
        t2: str,
        mbl: pd.DataFrame | None = None,
        save_fig: bool = False
    ) -> tuple[xr.DataArray,xr.DataArray] | tuple[xr.DataArray,xr.DataArray,xr.DataArray]:
    '''
    mbl = marine boundary layer dataframe

    Returns various background-time series for this given species
    '''

    ds_sel = ds_all_rem_out.sel(dataset=dataset_str, time=slice(t1, t2))

    fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    ax0 = axs[0]
    ax1 = axs[1]
    ds_sel.value.plot(marker=".", color="dimgrey", ls="", ax=ax0,zorder=0)
    percentile_10th = (
        ds_sel.value.rolling(time=mw, center=True).construct("tmp").quantile(quantile_value/100, dim="tmp")
    )
    plot_percentiles = percentile_10th.plot(
        marker=".", ls="-", color="k", lw=1,ax=ax0, label=f'moving {quantile_value}th percentile ({int(mw/24)}days)',zorder=2
    )  # take 3days windows and for each window the 10%-quantile
    # save the background data
    delta_to_background = ds_sel.value - percentile_10th

    # plot the NOAA fit for comparison
    filt, df_interp, ds_interp = run_curve_fit.run_ccgfilter(
                    ds=ds_sel.value,
                    dataset_str=dataset_str,
                    t1=t1,
                    t2=t2,
                    **fit_properties[dataset_str]
                )
    ds_sel['curve_fit'] = ds_interp["smoothed_vals"].resample(time='1h').mean()
    plot_noaa = ds_sel['curve_fit'].plot(ax=ax0,label='curve fit NOAA',color='orange',ls='--')
    # save the background data using NOAA
    delta_to_background_curvefit = ds_sel.value - ds_sel['curve_fit']

    
    # Plot the NOAA-MBL background (marine boundary layer)
    # It is only available for CO2 and CH4
    if isinstance(mbl,pd.DataFrame):
        mbl_background = mbl['value'].to_xarray().sel(time=slice(t1,t2))
        plot_mbl = mbl_background.plot(label='MBL',ax=ax0) # transform to xarray to have automatic time-axis alignment
        # save the background using MBL data as baseline
        delta_to_background_mbl =  ds_sel.value - mbl_background.resample(time='1h').interpolate() # interpolate the MBL data to 1h


    ## plot the background and the difference to this "background":
    # use percentiles as baseline:
    mask_is_background = ds_sel.value < percentile_10th
    #mask_is_background = mask_is_background_co ## use the CO background conditions

    # plot the "background" in light grey
    ds_sel.value[mask_is_background].plot(color='lightgrey',ls='',marker='.',ax=ax0,label='background percentiles',zorder=1)
    
    lgd0 = ax0.legend(bbox_to_anchor=(1.01, 0.5), loc='center left', ncol=1,fontsize=8)

    # ------ 2nd subplot ----- #
    ### plot data with removed background
    delta_to_background.where(delta_to_background > 0).plot(
        ax=ax1, color=plot_percentiles[0].get_color(),marker='.', label = 'removed percentile background'
    )  # plot all data that are larger than the "background" in new subplot
    delta_to_background_curvefit.where(delta_to_background_curvefit > 0).plot(
        ax=ax1, color=plot_noaa[0].get_color(),marker='.', label = 'removed curvefit background'
    )     
    
    if isinstance(mbl,pd.DataFrame):
        delta_to_background_mbl.where(delta_to_background_mbl > 0).plot(
            ax=ax1, color=plot_mbl[0].get_color(),marker='.', label = 'removed MBL background (lin. interp.)'
        ) 
    lgd1 = ax1.legend(bbox_to_anchor=(1.01, 0.5), loc='center left', ncol=1,fontsize=8)

    ax0.set_xlabel('')
    ax1.set_title("")
    ax1.set_title(f"Delta {dataset_str} (data-fitted background)", loc="left")
    plt.show()

    if save_fig: 
        fig_name = f"{dataset_str}_timeseries_with_backgroudns"
        plt.savefig(
        f"{dir_save}background/{fig_name}.png", dpi=300,bbox_extra_artists=(lgd0,lgd1), bbox_inches='tight'
        )

    if isinstance(mbl,pd.DataFrame):
        return delta_to_background, delta_to_background_mbl, delta_to_background_curvefit
    else: 
        return delta_to_background, delta_to_background_curvefit
    

In [ ]:
## Plot and get  backgrounds for CO2, using different background methods
dataset_str = "CO2"
delta_co2_quant, delta_co2_mbl, delta_co2_curvefit = plot_background(dataset_str,t1, t2, mbl_co2, save_fig=True)

## Plot and get  backgrounds for CH4, using different background methods
dataset_str = "CH4"
delta_ch4_quant, delta_ch4_mbl, delta_ch4_curvefit = plot_background(dataset_str,t1, t2, mbl = mbl_ch4, save_fig=True)


dataset_str = "CO"
delta_co_quant,  delta_co_curvefit = plot_background(dataset_str,t1, t2,save_fig=True)

### Check GFAS-derived CO data (footprint-weighted)

In [40]:
## Read in GFAS-derived flexpart CO: 
co_flexpart = xr.open_dataset(r'..\data\level3\flexpart\weighted_co_ts_values_2020-2023.nc')

In [ ]:
plt.subplots(figsize=(15,5))
#delta_co_curvefit.plot()
# use noaa-background or quantiles (delta_co_quant)

delta_co_curvefit.where(delta_co_curvefit > 0).plot(label='CO data above background (NOAA curve fit)') # Only use data ABOVE the curvefit/background??
co_flexpart['co_contribution'].plot(label='CO contribution from fires (flexpart)')
plt.legend()
plt.show()

In [ ]:
plt.subplots(figsize=(15,5))
#delta_co_curvefit.plot()
# use noaa-background or quantiles (delta_co_quant)

delta_co_quant.where(delta_co_quant > 0).plot(label='CO data above background (NOAA curve fit)') # Only use data ABOVE the curvefit/background??
co_flexpart['co_contribution'].plot(label='CO contribution from fires (flexpart)')
plt.legend()
plt.show()

In [43]:
# check day vs night data

mknlat, mknlon, mknalt = get_station_coords("MKN")  # Mt. Kenya station coordinates
sun_mkn = Sun(mknlat, mknlon)
# today_sr = sun.get_sunrise_time()
# today_ss = sun.get_sunset_time()
# print('Today at MKN the sun raised at {} and get down at {} UTC'.
#       format(today_sr.strftime('%H:%M'), today_ss.strftime('%H:%M')))


In [ ]:
sunrise_times = [sun_mkn.get_sunrise_time(to_datetime(t)) for t in delta_co2_curvefit.time.values]
# transform times to decimal times
sunrise_times_decimal = [t.hour + t.minute/60 for t in sunrise_times]

sunset_times = [sun_mkn.get_sunset_time(to_datetime(t)) for t in delta_co2_curvefit.time.values]
# transform times to decimal times
sunset_times_decimal = [t.hour + t.minute/60 for t in sunset_times]

In [ ]:
# make a pandas dataframe with sunrise and sunset times for each timestep
# set time as index
df = pd.DataFrame(index=pd.DatetimeIndex(pd.to_datetime(delta_co2_curvefit.time.values)))
df['time'] = [t.time() for t in df.index]
df['time_sunrise'] = [t.time() for t in sunrise_times]
df['time_sunset'] = [t.time() for t in sunset_times]
df['is_day'] = (df['time'] > df['time_sunrise']) & (df['time'] < df['time_sunset'])
df

In [46]:
dates_day = df.index.where(df['is_day']).dropna().values
dates_night = df.index.where(df['is_day']==False).dropna().values
day_data = delta_co_curvefit.where(delta_co_curvefit['time'].isin(dates_day))
night_data = delta_co_curvefit.where(delta_co_curvefit['time'].isin(dates_day)==False)

In [ ]:
plt.subplots(figsize=(15,5))
day_data.where(day_data > 0).plot(label='CO during day above background (NOAA curve fit)')
night_data.where(night_data > 0).plot(label='CO during night above background (NOAA curve fit)') # Only use data ABOVE the curvefit/background??
co_flexpart['co_contribution'].plot(label='CO contribution from fires (flexpart)')
plt.legend()
plt.show()

In [ ]:

co_flexpart

In [ ]:
# scatterplot for night data only
import seaborn as sns
import scipy 
night_data = delta_co_curvefit.where(delta_co_curvefit['time'].isin(dates_day)==False,drop=True)

data_to_plot = night_data
#data_to_plot = delta_co_curvefit

x = co_flexpart['co_contribution'].resample(release_time='D').mean().values
y = data_to_plot.where(data_to_plot>0).resample(time='D').mean().values
plt.figure()
p = sns.regplot(x=x,y=y)
plt.show()

#calculate slope and intercept of regression equation
slope, intercept, r, p, sterr = scipy.stats.linregress(x=p.get_lines()[0].get_xdata(),
                                                       y=p.get_lines()[0].get_ydata())

print(f"Regression equation: y = {slope:.2f}x + {intercept:.2f}, r = {r:.2f}, p = {p:.2f}, sterr = {sterr:.2f}")

In [ ]:
# scatter plot for July 2021
night_data = delta_co_curvefit.where(delta_co_curvefit['time'].isin(dates_day)==False,drop=True)

data_to_plot = night_data
data_to_plot = delta_co_curvefit.sel(time=slice('2021-07-15','2021-07-31'))

x = co_flexpart['co_contribution'].sel(release_time=slice('2021-07-15','2021-07-31')).resample(release_time='D').mean().values
y = data_to_plot.where(data_to_plot>0).resample(time='D').mean().values
plt.figure()
p = sns.regplot(x=x,y=y)
plt.show()

#calculate slope and intercept of regression equation
slope, intercept, r, p, sterr = scipy.stats.linregress(x=p.get_lines()[0].get_xdata(),
                                                       y=p.get_lines()[0].get_ydata())

print(f"Regression equation: y = {slope:.2f}x + {intercept:.2f}, r = {r:.2f}, p = {p:.2f}, sterr = {sterr:.2f}")

In [ ]:
## Compare CO and CO2 deltas (removed background)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

sampling = '1D'
# x= delta_co2_quant.where(delta_co2_quant > 0).resample(time=sampling).mean()
# y= delta_co_quant.where(delta_co_quant > 0).resample(time=sampling).mean()
# compare noaa backgrounds:
x= delta_co2_curvefit.where(delta_co2_curvefit > 0)
y= delta_co_curvefit.where(delta_co_curvefit > 0)

#compare CO2-MBL background with CO percentile background:
# x= delta_co2_mbl
# y= delta_co.sel(time=slice(x.time[0],x.time[-1]))

months = x.time.dt.month

# Define a colormap for months
colmap = tol_colors.tol_cmap("rainbow_discrete", 12)  # 12 colors for 12 months
colors = colmap(np.linspace(0, 1, 12))

fig, ax = plt.subplots(layout='constrained')
scatter = ax.scatter(x, y, c=months, cmap=colmap, alpha=1,s=3)
## add line fits:
# Iterate over unique months and fit a line for each month
for month, c in zip(range(1, 13), colors):
    # Filter data for the current month
    x_month = x.where(x.time.dt.month == month, drop=True)
    y_month = y.where(y.time.dt.month == month, drop=True)

    # data may contain some nans, remove them to be able to make a fit
    y_month_non_nan = y_month.dropna(dim="time")
    x_month_non_nan = x_month.sel(time=y_month_non_nan.time.values, drop=True) #select corresponding data
    # other way around
    x_month_non_nan = x_month_non_nan.dropna(dim="time")
    y_month_non_nan = y_month.sel(time=x_month_non_nan.time.values, drop=True) #select corresponding data

    if len(x_month) > 1:
        # Fit a linear regression model
        model = LinearRegression()
        model.fit(x_month_non_nan.values.reshape(-1, 1), y_month_non_nan.values)

        # Predictions
        x_pred = np.linspace(
            x.min(), x.max(), 2
        )  # create x-values for which to predict y-values
        y_pred = model.predict(x_pred.reshape(-1, 1))

        # Plot the line fit for the current month
        r2 = model.score(x_month_non_nan.values.reshape(-1, 1), y_month_non_nan.values)
        slope = model.coef_[0]
        month_str = dt.datetime.strptime(str(month), "%m").strftime("%b")
        ax.plot(
            x_pred, y_pred, color=c, label=f"{month_str} ({r2:.2f}, {slope:.2f})"
        ) 
    else:
        print(f"no fit for month {month}")
ax.set_ylabel(f"Delta CO (ppb)")
ax.set_xlabel("Delta CO2 (ppm)")

# general figure properties
plt.setp(ax, box_aspect=1)  # get quadratic plots
legend1 = ax.legend(
    #*scatter.legend_elements(), 
    loc="upper left", title="Months (R2, slope)",
    bbox_to_anchor=(1,1)
)
ax.add_artist(legend1)
plt.autoscale(tight=True)
plt.show()


fig_name = f'scatter_deltaCO_deltaCO2_{sampling}'
#fig_name = 'scatter_deltaCO_deltaCO2_NOAAbckgr'
plt.savefig(f"{dir_save}background/{fig_name}.png", 
    bbox_extra_artists=(legend1,), bbox_inches='tight')


In [ ]:
df = x.to_dataframe(name='delta_co2')
df = pd.concat([df,y.to_dataframe(name='delta_co')])
df['month'] = df.index.month
#df['season'] = df.index.season
 
import seaborn as sns
plt.figure()
sns.scatterplot(df,x='delta_co2',y='delta_co',hue='month',legend='full')
plt.show()

In [26]:
# # same with seaborn (but without regression line)
# df = x.to_dataframe(name='delta_co2')
# df = pd.concat([df,y.to_dataframe(name='delta_co')])
# df['month'] = df.index.month
# #df['season'] = df.index.season
 
# import seaborn as sns
# plt.figure()
# sns.scatterplot(df,x='delta_co2',y='delta_co',hue='month',legend='full')
# plt.show()

In [ ]:
ds_co = ds_sel = ds_all.sel(dataset='CO', time=slice(t1, t2))
ds_co

In [ ]:
plt.figure()
ds_co.where(ds_co['value']>80)['value'].plot()
plt.show()